In [1]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import psycopg2
from psycopg2 import sql

from IPython.display import display

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nEnvironment verification completed.")

Python executable:
C:\Users\asus\anaconda3\python.exe

Python version:
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]

Environment verification completed.


In [3]:
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "ecommerce_ai_db",
    "user": "postgres"
}

print("Database configuration loaded.")
print(f"Host: {DB_CONFIG['host']}")
print(f"Port: {DB_CONFIG['port']}")
print(f"Database: {DB_CONFIG['database']}")
print(f"User: {DB_CONFIG['user']}")

Database configuration loaded.
Host: localhost
Port: 5432
Database: ecommerce_ai_db
User: postgres


In [4]:
from getpass import getpass

DB_PASSWORD = getpass("Enter your PostgreSQL password: ")

conn = psycopg2.connect(
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"],
    database=DB_CONFIG["database"],
    user=DB_CONFIG["user"],
    password=DB_PASSWORD
)

print("Connected successfully to PostgreSQL!")

Enter your PostgreSQL password:  ········


Connected successfully to PostgreSQL!


In [5]:
database_check_query = """
SELECT
    current_database() AS database_name,
    current_user AS user_name,
    version() AS postgres_version;
"""

database_info = pd.read_sql_query(
    database_check_query,
    conn
)

display(database_info)

,database_name,user_name,postgres_version
0,ecommerce_ai_db,postgres,"PostgreSQL 18.4 on x86_64-windows, compiled by..."


In [6]:
tables_query = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
"""

tables_df = pd.read_sql_query(
    tables_query,
    conn
)

display(tables_df)

,table_schema,table_name
0,public,category_translation
1,public,customers
2,public,geolocation
3,public,order_items
4,public,order_payments
5,public,order_reviews
6,public,orders
7,public,products
8,public,sellers


In [7]:
expected_tables = {
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
}

actual_tables = set(tables_df["table_name"])

missing_tables = expected_tables - actual_tables

extra_tables = actual_tables - expected_tables

print("Expected tables:")
print(expected_tables)

print("\nActual tables:")
print(actual_tables)

print("\nMissing expected tables:")
print(missing_tables)

print("\nAdditional tables:")
print(extra_tables)

Expected tables:
{'order_payments', 'products', 'order_reviews', 'orders', 'sellers', 'order_items', 'customers', 'category_translation', 'geolocation'}

Actual tables:
{'order_payments', 'products', 'order_reviews', 'orders', 'sellers', 'order_items', 'customers', 'category_translation', 'geolocation'}

Missing expected tables:
set()

Additional tables:
set()


In [8]:
if missing_tables:
    raise ValueError(
        f"Required tables are missing from PostgreSQL: {missing_tables}"
    )

print("All required tables are available.")

All required tables are available.


In [9]:
columns_query = """
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    udt_name,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY
    table_name,
    ordinal_position;
"""

columns_df = pd.read_sql_query(
    columns_query,
    conn
)

display(columns_df)

,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable
0,public,category_translation,1,product_category_name,text,text,YES
1,public,category_translation,2,product_category_name_english,text,text,YES
2,public,customers,1,customer_id,text,text,YES
3,public,customers,2,customer_unique_id,text,text,YES
4,public,customers,3,customer_zip_code_prefix,bigint,int8,YES
...,...,...,...,...,...,...,...
56,public,products,10,product_category_name_english,text,text,YES
57,public,sellers,1,seller_id,text,text,YES
58,public,sellers,2,seller_zip_code_prefix,bigint,int8,YES
59,public,sellers,3,seller_city,text,text,YES


In [10]:
schema_summary = (
    columns_df
    .groupby("table_name")
    .agg(
        column_count=("column_name", "count"),
        columns=("column_name", lambda values: ", ".join(values))
    )
    .reset_index()
)

display(schema_summary)

,table_name,column_count,columns
0,category_translation,2,"product_category_name, product_category_name_e..."
1,customers,5,"customer_id, customer_unique_id, customer_zip_..."
2,geolocation,5,"geolocation_zip_code_prefix, geolocation_lat, ..."
3,order_items,7,"order_id, order_item_id, product_id, seller_id..."
4,order_payments,5,"order_id, payment_sequential, payment_type, pa..."
5,order_reviews,7,"review_id, order_id, review_score, review_comm..."
6,orders,16,"order_id, customer_id, order_status, order_pur..."
7,products,10,"product_id, product_category_name, product_nam..."
8,sellers,4,"seller_id, seller_zip_code_prefix, seller_city..."


In [11]:
required_columns = {
    "orders": {
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    },
    "order_items": {
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    },
    "order_payments": {
        "order_id",
        "payment_type",
        "payment_value"
    },
    "customers": {
        "customer_id",
        "customer_unique_id"
    },
    "products": {
        "product_id",
        "product_category_name"
    },
    "sellers": {
        "seller_id"
    },
    "order_reviews": {
        "order_id",
        "review_score"
    }
}

for table_name, required_column_set in required_columns.items():

    actual_columns = set(
        columns_df.loc[
            columns_df["table_name"] == table_name,
            "column_name"
        ]
    )

    missing_columns = required_column_set - actual_columns

    if missing_columns:
        raise ValueError(
            f"Missing columns in table '{table_name}': {missing_columns}"
        )

print("All required analysis columns are available.")

All required analysis columns are available.


In [12]:
important_columns = columns_df[
    (
        columns_df["table_name"].isin(
            [
                "orders",
                "order_items",
                "order_payments",
                "customers",
                "products",
                "sellers",
                "order_reviews"
            ]
        )
    )
    &
    (
        columns_df["column_name"].isin(
            [
                "order_id",
                "customer_id",
                "customer_unique_id",
                "product_id",
                "seller_id",
                "order_status",
                "order_purchase_timestamp",
                "price",
                "freight_value",
                "payment_value",
                "review_score"
            ]
        )
    )
]

display(important_columns)

,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable
2,public,customers,1,customer_id,text,text,YES
3,public,customers,2,customer_unique_id,text,text,YES
12,public,order_items,1,order_id,text,text,YES
14,public,order_items,3,product_id,text,text,YES
15,public,order_items,4,seller_id,text,text,YES
17,public,order_items,6,price,double precision,float8,YES
18,public,order_items,7,freight_value,double precision,float8,YES
19,public,order_payments,1,order_id,text,text,YES
23,public,order_payments,5,payment_value,double precision,float8,YES
25,public,order_reviews,2,order_id,text,text,YES


In [13]:
row_count_query = """
SELECT
    'customers' AS table_name,
    COUNT(*) AS row_count
FROM public.customers

UNION ALL

SELECT
    'geolocation',
    COUNT(*)
FROM public.geolocation

UNION ALL

SELECT
    'order_items',
    COUNT(*)
FROM public.order_items

UNION ALL

SELECT
    'order_payments',
    COUNT(*)
FROM public.order_payments

UNION ALL

SELECT
    'order_reviews',
    COUNT(*)
FROM public.order_reviews

UNION ALL

SELECT
    'orders',
    COUNT(*)
FROM public.orders

UNION ALL

SELECT
    'products',
    COUNT(*)
FROM public.products

UNION ALL

SELECT
    'sellers',
    COUNT(*)
FROM public.sellers

UNION ALL

SELECT
    'category_translation',
    COUNT(*)
FROM public.category_translation

ORDER BY table_name;
"""

row_counts_df = pd.read_sql_query(
    row_count_query,
    conn
)

display(row_counts_df)

,table_name,row_count
0,category_translation,71
1,customers,99441
2,geolocation,738332
3,order_items,112650
4,order_payments,103886
5,order_reviews,99224
6,orders,99441
7,products,32951
8,sellers,3095


# Business Analysis

The following sections analyze:

1. Overall sales performance
2. Monthly sales trends
3. Order status performance
4. Top products
5. Top product categories
6. Seller performance
7. Customer behavior
8. Payment methods
9. Customer review scores
10. Delivery performance
11. Geographic performance

In [15]:
overall_performance_query = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    COUNT(DISTINCT oi.product_id) AS total_products,
    COUNT(DISTINCT oi.seller_id) AS total_sellers,
    SUM(CAST(oi.price AS NUMERIC)) AS total_product_revenue,
    SUM(CAST(oi.freight_value AS NUMERIC)) AS total_freight_revenue,
    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_order_value
FROM public.orders AS o
LEFT JOIN public.order_items AS oi
    ON o.order_id = oi.order_id;
"""

overall_performance_df = pd.read_sql_query(
    overall_performance_query,
    conn
)

display(overall_performance_df)

,total_orders,total_customers,total_products,total_sellers,total_product_revenue,total_freight_revenue,total_order_value
0,99441,99441,32951,3095,13591643.7,2251909.54,15843553.24


In [16]:
monthly_sales_query = """
SELECT
    DATE_TRUNC(
        'month',
        CAST(o.order_purchase_timestamp AS TIMESTAMP)
    ) AS sales_month,

    COUNT(DISTINCT o.order_id) AS total_orders,

    SUM(CAST(oi.price AS NUMERIC)) AS product_revenue,

    SUM(CAST(oi.freight_value AS NUMERIC)) AS freight_revenue,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_revenue

FROM public.orders AS o

INNER JOIN public.order_items AS oi
    ON o.order_id = oi.order_id

WHERE o.order_purchase_timestamp IS NOT NULL

GROUP BY
    DATE_TRUNC(
        'month',
        CAST(o.order_purchase_timestamp AS TIMESTAMP)
    )

ORDER BY sales_month;
"""

monthly_sales_df = pd.read_sql_query(
    monthly_sales_query,
    conn
)

display(monthly_sales_df)

,sales_month,total_orders,product_revenue,freight_revenue,total_revenue
0,2016-09-01,3,267.36,87.39,354.75
1,2016-10-01,308,49507.66,7301.18,56808.84
2,2016-12-01,1,10.90,8.72,19.62
3,2017-01-01,789,120312.87,16875.62,137188.49
4,2017-02-01,1733,247303.02,38977.60,286280.62
5,2017-03-01,2641,374344.30,57704.29,432048.59
6,2017-04-01,2391,359927.23,52495.01,412422.24
7,2017-05-01,3660,506071.14,80119.81,586190.95
8,2017-06-01,3217,433038.60,69924.44,502963.04
9,2017-07-01,3969,498031.48,86940.14,584971.62


In [17]:
monthly_sales_df["revenue_growth_percentage"] = (
    monthly_sales_df["total_revenue"]
    .pct_change()
    .mul(100)
)

display(monthly_sales_df)

,sales_month,total_orders,product_revenue,freight_revenue,total_revenue,revenue_growth_percentage
0,2016-09-01,3,267.36,87.39,354.75,NaN
1,2016-10-01,308,49507.66,7301.18,56808.84,15913.767442
2,2016-12-01,1,10.90,8.72,19.62,-99.965463
3,2017-01-01,789,120312.87,16875.62,137188.49,699127.777778
4,2017-02-01,1733,247303.02,38977.60,286280.62,108.676850
5,2017-03-01,2641,374344.30,57704.29,432048.59,50.917862
6,2017-04-01,2391,359927.23,52495.01,412422.24,-4.542626
7,2017-05-01,3660,506071.14,80119.81,586190.95,42.133690
8,2017-06-01,3217,433038.60,69924.44,502963.04,-14.198089
9,2017-07-01,3969,498031.48,86940.14,584971.62,16.305091


In [18]:
order_status_query = """
SELECT
    o.order_status,

    COUNT(DISTINCT o.order_id) AS total_orders,

    ROUND(
        COUNT(DISTINCT o.order_id)::NUMERIC
        * 100
        /
        NULLIF(
            SUM(COUNT(DISTINCT o.order_id))
            OVER (),
            0
        ),
        2
    ) AS order_percentage

FROM public.orders AS o

GROUP BY
    o.order_status

ORDER BY
    total_orders DESC;
"""

order_status_df = pd.read_sql_query(
    order_status_query,
    conn
)

display(order_status_df)

,order_status,total_orders,order_percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [19]:
top_products_query = """
SELECT
    oi.product_id,

    COUNT(DISTINCT oi.order_id) AS total_orders,

    SUM(CAST(oi.price AS NUMERIC)) AS product_revenue,

    SUM(CAST(oi.freight_value AS NUMERIC)) AS freight_revenue,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_revenue

FROM public.order_items AS oi

GROUP BY
    oi.product_id

ORDER BY
    total_revenue DESC

LIMIT 20;
"""

top_products_df = pd.read_sql_query(
    top_products_query,
    conn
)

display(top_products_df)

,product_id,total_orders,product_revenue,freight_revenue,total_revenue
0,bb50f2e236e5eea0100680137654686c,187,63885.00,3721.10,67606.10
1,d1c427060a0f73f6b889a5c7c61f2ac4,323,47214.51,13761.52,60976.03
2,6cdd53843498f92890544667809f1595,151,54730.20,4363.79,59093.99
3,99a4788cb24856965c36a24e339b6058,467,43025.56,8046.04,51071.60
4,d6160fb7873f184099d9bc95e30376af,35,48899.34,1426.84,50326.18
5,3dd2a17168ec895c781a9191c1e95ad7,255,41082.60,7129.62,48212.22
6,aca2eb7d00ea1a7b8ebd4e68314663af,431,37608.90,7211.86,44820.76
7,5f504b3a1c75b73d6151be81eb05bdc9,63,37733.90,3991.91,41725.81
8,25c38557cf793876c5abdd5931f922db,38,38907.32,1404.63,40311.95
9,53b36df67ebb7c41585e8d54d6772e08,306,37683.42,2274.51,39957.93


In [20]:
category_sales_query = """
SELECT
    COALESCE(
        NULLIF(
            TRIM(p.product_category_name),
            ''
        ),
        'Unknown'
    ) AS product_category,

    COUNT(DISTINCT oi.order_id) AS total_orders,

    COUNT(oi.product_id) AS total_items_sold,

    SUM(CAST(oi.price AS NUMERIC)) AS product_revenue,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_revenue

FROM public.order_items AS oi

LEFT JOIN public.products AS p
    ON oi.product_id = p.product_id

GROUP BY
    COALESCE(
        NULLIF(
            TRIM(p.product_category_name),
            ''
        ),
        'Unknown'
    )

ORDER BY
    total_revenue DESC

LIMIT 20;
"""

category_sales_df = pd.read_sql_query(
    category_sales_query,
    conn
)

display(category_sales_df)

,product_category,total_orders,total_items_sold,product_revenue,total_revenue
0,beleza_saude,8836,9670,1258681.34,1441248.07
1,relogios_presentes,5624,5991,1205005.68,1305541.61
2,cama_mesa_banho,9417,11115,1036988.68,1241681.72
3,esporte_lazer,7720,8641,988048.97,1156656.48
4,informatica_acessorios,6689,7827,911954.32,1059272.40
5,moveis_decoracao,6449,8334,729762.49,902511.79
6,utilidades_domesticas,5884,6964,632248.66,778397.77
7,cool_stuff,3632,3796,635290.85,719329.95
8,automotivo,3897,4235,592720.11,685384.32
9,ferramentas_jardim,3518,4347,485256.46,584219.21


In [21]:
seller_performance_query = """
SELECT
    oi.seller_id,

    COUNT(DISTINCT oi.order_id) AS total_orders,

    COUNT(oi.product_id) AS total_items_sold,

    SUM(CAST(oi.price AS NUMERIC)) AS product_revenue,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_revenue

FROM public.order_items AS oi

GROUP BY
    oi.seller_id

ORDER BY
    total_revenue DESC

LIMIT 20;
"""

seller_performance_df = pd.read_sql_query(
    seller_performance_query,
    conn
)

display(seller_performance_df)

,seller_id,total_orders,total_items_sold,product_revenue,total_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,1132,1156,229472.63,249640.70
1,7c67e1448b00f6e969d365cea6b010ab,982,1364,187923.89,239536.44
2,53243585a1d6dc2643021fd1853d8905,358,410,222776.05,235856.68
3,4a3ca9315b744ce9f8e9374361493884,1806,1987,200472.92,235539.96
4,fa1c13f2614d7b5c4749cbc52fecda94,585,586,194042.03,204084.73
5,da8622b14eb17ae2831f4ac5b9dab84a,1314,1551,160236.57,185192.32
6,7e93a43ef30c4f03f38b393420bc753a,336,340,176431.87,182754.05
7,1025f0e2d44d7041d6cf58b6550e0bfa,915,1428,138968.55,172860.69
8,7a67c85e85bb2ce8582c35f2203ad736,1160,1171,141745.53,162648.38
9,955fee9216a65b617aa5c0531780ce60,1287,1499,135171.70,160602.68


In [22]:
customer_purchase_query = """
SELECT
    c.customer_unique_id,

    COUNT(DISTINCT o.order_id) AS total_orders,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_spent,

    AVG(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS average_item_order_value

FROM public.customers AS c

INNER JOIN public.orders AS o
    ON c.customer_id = o.customer_id

INNER JOIN public.order_items AS oi
    ON o.order_id = oi.order_id

GROUP BY
    c.customer_unique_id

ORDER BY
    total_spent DESC

LIMIT 20;
"""

customer_purchase_df = pd.read_sql_query(
    customer_purchase_query,
    conn
)

display(customer_purchase_df)

,customer_unique_id,total_orders,total_spent,average_item_order_value
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,1708.010
1,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.815
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,1818.720
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.310
4,459bef486812aa25204be022145caa62,1,6922.21,6922.210
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.660
6,4007669dec559734d6f53e029e360987,1,6081.54,1013.590
7,5d0a2980b292d049061542014e8960bf,1,4809.44,2404.720
8,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.340
9,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.780


In [23]:
repeat_customer_query = """
WITH customer_orders AS (

    SELECT
        c.customer_unique_id,

        COUNT(DISTINCT o.order_id) AS order_count

    FROM public.customers AS c

    INNER JOIN public.orders AS o
        ON c.customer_id = o.customer_id

    GROUP BY
        c.customer_unique_id
)

SELECT
    CASE
        WHEN order_count = 1
            THEN 'One-time customer'

        WHEN order_count > 1
            THEN 'Repeat customer'

        ELSE 'Unknown'

    END AS customer_type,

    COUNT(*) AS customer_count

FROM customer_orders

GROUP BY
    CASE
        WHEN order_count = 1
            THEN 'One-time customer'

        WHEN order_count > 1
            THEN 'Repeat customer'

        ELSE 'Unknown'

    END

ORDER BY
    customer_count DESC;
"""

repeat_customer_df = pd.read_sql_query(
    repeat_customer_query,
    conn
)

display(repeat_customer_df)

,customer_type,customer_count
0,One-time customer,93099
1,Repeat customer,2997


In [24]:
payment_method_query = """
SELECT
    op.payment_type,

    COUNT(DISTINCT op.order_id) AS total_orders,

    COUNT(op.payment_type) AS payment_transactions,

    SUM(
        CAST(op.payment_value AS NUMERIC)
    ) AS total_payment_value,

    AVG(
        CAST(op.payment_value AS NUMERIC)
    ) AS average_payment_value

FROM public.order_payments AS op

GROUP BY
    op.payment_type

ORDER BY
    total_payment_value DESC;
"""

payment_method_df = pd.read_sql_query(
    payment_method_query,
    conn
)

display(payment_method_df)

,payment_type,total_orders,payment_transactions,total_payment_value,average_payment_value
0,credit_card,76505,76795,12542084.19,163.319021
1,boleto,19784,19784,2869361.27,145.034435
2,voucher,3866,5775,379436.87,65.703354
3,debit_card,1528,1529,217989.79,142.570170
4,not_defined,3,3,0.00,0.000000


In [25]:
review_score_query = """
SELECT
    CAST(review_score AS INTEGER) AS review_score,

    COUNT(*) AS review_count,

    ROUND(
        COUNT(*)::NUMERIC
        * 100
        /
        NULLIF(
            SUM(COUNT(*))
            OVER (),
            0
        ),
        2
    ) AS review_percentage

FROM public.order_reviews

WHERE review_score IS NOT NULL

GROUP BY
    CAST(review_score AS INTEGER)

ORDER BY
    review_score;
"""

review_score_df = pd.read_sql_query(
    review_score_query,
    conn
)

display(review_score_df)

,review_score,review_count,review_percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [26]:
average_review_query = """
SELECT
    ROUND(
        AVG(
            CAST(review_score AS NUMERIC)
        ),
        2
    ) AS average_review_score,

    COUNT(*) AS total_reviews

FROM public.order_reviews

WHERE review_score IS NOT NULL;
"""

average_review_df = pd.read_sql_query(
    average_review_query,
    conn
)

display(average_review_df)

,average_review_score,total_reviews
0,4.09,99224


In [27]:
delivery_performance_query = """
SELECT

    AVG(
        CAST(
            CAST(order_delivered_customer_date AS TIMESTAMP)
            -
            CAST(order_purchase_timestamp AS TIMESTAMP)
            AS INTERVAL
        )
    ) AS average_delivery_time,

    COUNT(*) AS delivered_orders

FROM public.orders

WHERE
    order_delivered_customer_date IS NOT NULL

    AND order_purchase_timestamp IS NOT NULL

    AND CAST(order_delivered_customer_date AS TIMESTAMP)
        >=
        CAST(order_purchase_timestamp AS TIMESTAMP);
"""

delivery_performance_df = pd.read_sql_query(
    delivery_performance_query,
    conn
)

display(delivery_performance_df)

,average_delivery_time,delivered_orders
0,12 days 22:07:28.205640,99441


In [28]:
delivery_days_query = """
SELECT

    ROUND(
        AVG(
            EXTRACT(
                EPOCH FROM
                (
                    CAST(order_delivered_customer_date AS TIMESTAMP)
                    -
                    CAST(order_purchase_timestamp AS TIMESTAMP)
                )
            ) / 86400
        ),
        2
    ) AS average_delivery_days,

    MIN(
        EXTRACT(
            EPOCH FROM
            (
                CAST(order_delivered_customer_date AS TIMESTAMP)
                -
                CAST(order_purchase_timestamp AS TIMESTAMP)
            )
        ) / 86400
    ) AS minimum_delivery_days,

    MAX(
        EXTRACT(
            EPOCH FROM
            (
                CAST(order_delivered_customer_date AS TIMESTAMP)
                -
                CAST(order_purchase_timestamp AS TIMESTAMP)
            )
        ) / 86400
    ) AS maximum_delivery_days

FROM public.orders

WHERE
    order_delivered_customer_date IS NOT NULL

    AND order_purchase_timestamp IS NOT NULL

    AND CAST(order_delivered_customer_date AS TIMESTAMP)
        >=
        CAST(order_purchase_timestamp AS TIMESTAMP);
"""

delivery_days_df = pd.read_sql_query(
    delivery_days_query,
    conn
)

display(delivery_days_df)

,average_delivery_days,minimum_delivery_days,maximum_delivery_days
0,12.92,0.533414,209.628611


In [29]:
customer_state_query = """
SELECT
    c.customer_state,

    COUNT(DISTINCT c.customer_unique_id)
        AS total_customers,

    COUNT(DISTINCT o.order_id)
        AS total_orders,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_revenue

FROM public.customers AS c

INNER JOIN public.orders AS o
    ON c.customer_id = o.customer_id

INNER JOIN public.order_items AS oi
    ON o.order_id = oi.order_id

GROUP BY
    c.customer_state

ORDER BY
    total_revenue DESC;
"""

customer_state_df = pd.read_sql_query(
    customer_state_query,
    conn
)

display(customer_state_df)

,customer_state,total_customers,total_orders,total_revenue
0,SP,39981,41375,5921678.12
1,RJ,12303,12762,2129681.98
2,MG,11178,11544,1856161.49
3,RS,5249,5432,885826.76
4,PR,4840,4998,800935.44
5,BA,3257,3358,611506.67
6,SC,3513,3612,610213.60
7,DF,2062,2125,353229.44
8,GO,1942,2007,347706.93
9,ES,1956,2025,324801.91


In [30]:
seller_state_query = """
SELECT
    s.seller_state,

    COUNT(DISTINCT s.seller_id)
        AS total_sellers,

    COUNT(DISTINCT oi.order_id)
        AS total_orders,

    SUM(
        CAST(oi.price AS NUMERIC)
        + CAST(oi.freight_value AS NUMERIC)
    ) AS total_revenue

FROM public.sellers AS s

INNER JOIN public.order_items AS oi
    ON s.seller_id = oi.seller_id

GROUP BY
    s.seller_state

ORDER BY
    total_revenue DESC;
"""

seller_state_df = pd.read_sql_query(
    seller_state_query,
    conn
)

display(seller_state_df)

,seller_state,total_sellers,total_orders,total_revenue
0,SP,1849,70188,10235883.88
1,PR,349,7673,1458900.73
2,MG,244,7930,1224159.80
3,RJ,171,4353,937814.12
4,SC,190,3667,738973.13
5,RS,129,1989,435802.63
6,BA,19,569,305262.24
7,DF,30,824,116243.54
8,PE,9,406,103886.31
9,GO,40,463,78964.71


In [31]:
data_quality_query = """
SELECT
    'orders.order_id' AS field_name,
    COUNT(*) AS total_rows,
    COUNT(order_id) AS non_null_rows,
    COUNT(*) - COUNT(order_id) AS null_rows
FROM public.orders

UNION ALL

SELECT
    'orders.customer_id',
    COUNT(*),
    COUNT(customer_id),
    COUNT(*) - COUNT(customer_id)
FROM public.orders

UNION ALL

SELECT
    'order_items.product_id',
    COUNT(*),
    COUNT(product_id),
    COUNT(*) - COUNT(product_id)
FROM public.order_items

UNION ALL

SELECT
    'order_items.price',
    COUNT(*),
    COUNT(price),
    COUNT(*) - COUNT(price)
FROM public.order_items

UNION ALL

SELECT
    'order_payments.payment_value',
    COUNT(*),
    COUNT(payment_value),
    COUNT(*) - COUNT(payment_value)
FROM public.order_payments

UNION ALL

SELECT
    'order_reviews.review_score',
    COUNT(*),
    COUNT(review_score),
    COUNT(*) - COUNT(review_score)
FROM public.order_reviews;
"""

data_quality_df = pd.read_sql_query(
    data_quality_query,
    conn
)

display(data_quality_df)

,field_name,total_rows,non_null_rows,null_rows
0,order_payments.payment_value,103886,103886,0
1,order_reviews.review_score,99224,99224,0
2,order_items.price,112650,112650,0
3,order_items.product_id,112650,112650,0
4,orders.order_id,99441,99441,0
5,orders.customer_id,99441,99441,0


In [32]:
analysis_summary = {
    "total_orders": int(
        overall_performance_df.loc[
            0,
            "total_orders"
        ]
    ),

    "total_customers": int(
        overall_performance_df.loc[
            0,
            "total_customers"
        ]
    ),

    "total_products": int(
        overall_performance_df.loc[
            0,
            "total_products"
        ]
    ),

    "total_sellers": int(
        overall_performance_df.loc[
            0,
            "total_sellers"
        ]
    ),

    "total_revenue": float(
        overall_performance_df.loc[
            0,
            "total_order_value"
        ]
    )
}

analysis_summary_df = pd.DataFrame(
    [analysis_summary]
)

display(analysis_summary_df)

,total_orders,total_customers,total_products,total_sellers,total_revenue
0,99441,99441,32951,3095,15843553.24


In [33]:
if conn.closed == 0:

    conn.close()

    print("PostgreSQL connection closed successfully.")

else:

    print("PostgreSQL connection was already closed.")

PostgreSQL connection closed successfully.
